# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это ТРЕТИЙ шаг. На прошлом шаге 17e_study_23.2b агент научился заходить в иглу на ночном уровне с медведем. Здесь уже просто отпускаем агента в свободное плавание - пусть учиться играть без костылей.

В отличие от 22 серии здесь используется агент, у которого нет отжига по tau.

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 5
    generation_ind = 2
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    ####
    
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    HP.env.idle_penalty = 0
    HP.env.life_lost_penalty = 0

    HP.agent.parent = optuna_trial.suggest_categorical('agent.parent', [
        '17e_ppo_tr_atari_mp_16:59',
        '17e_ppo_tr_atari_mp_16:69',
        '17e_ppo_tr_atari_mp_16:70',
    ])
    HP.agent.observation_space_shape = (3, 84, 84) 
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.cnn_expand_dim = 1024 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = None
    HP.video.capture_env_ram_patches = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1_101:1', 
        'com.develorium.neurolab.frostbite_ram:level4_101:1', # bear level, day
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'three_lives', 'no_igloo'],
    ]

    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 16 # obs shape (3, 84, 84) -> 16, obs shape (1, 84, 84) -> 8
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

# Results
<TBD>

Результат прогона даже хуже, чем в 22-ой серии. Видно, что более стабильно по нижней границе, но нет прорывов.

<img src="./img/levels_passed.png" width=700><img src="./img/levels_passed_22.3a.png" width=700>
<img src="./img/reward.png" width=700><img src="./img/reward_22.3a.png" width=700>

Градиенты тоже не лучше...

*23*

<img src="./img/grad_norms.png">

*22*

<img src="./img/grad_norms_22.3a.png">

Такое ощущение, что на этом шаге произошло одеревенение...

**Выводы**
1) попробовать прогнать с более высокими learning_rate и ent_coef